# Tensión arterial: ¿quitarla o rellenarla?

La app no pregunta la tensión y las analíticas casi nunca la traen. La usaban los niveles 2 (`extras`) y 4 (`full`). Se comparan tres opciones:

| Opción | Qué es |
|---|---|
| `real` | El modelo de entonces, con la tensión de cada persona: lo mejor posible |
| `filled` | El modelo de entonces, sin la tensión, que se rellena con la mediana de su sexo y grupo de edad: lo que habría hecho la app |
| `removed` | El modelo entrenado sin tensión |

Todo con los mismos 5 bloques dentro de train; el test no se toca. Listón, el mismo que para quitar variables en el nivel 2: una opción vale si baja como mucho **0,002** el C-index misma edad frente a `real`.

In [1]:
# Imports
import numpy as np
import pandas as pd

from bioage.config.model_levels import LEVEL_COLUMNS
from bioage.config.paths import GOLD_TABLE
from bioage.modeling.s1_split import split_data
from bioage.modeling.s3_algorithms import build_pipeline, survival_target
from bioage.modeling.s4_biological_age import biological_age, fit_reference, risk_at_horizon
from bioage.modeling.s5_evaluation import c_index_by_group, calibration_table, same_age_c_index
from bioage.utils.features import age_groups
from bioage.utils.notebook_helpers import without

C:\Data\1_MisCosas\Evolve\TFM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BLOOD_PRESSURE = ["systolic_bp", "diastolic_bp"]
LEVELS = ["extras", "full"]
OPTIONS = ["real", "filled", "removed"]

gold = pd.read_parquet(GOLD_TABLE)
train, test, folds = split_data(gold)
print(f"Train: {len(train)} people; with blood pressure: {train[BLOOD_PRESSURE].notna().all(axis=1).mean():.1%}")

Train: 38795 people; with blood pressure: 95.7%


## 1. Riesgo y edad biológica con cada opción

Cada persona se evalúa con un modelo entrenado sin ella, y la curva de referencia sale de los mismos bloques.

In [3]:
def out_of_fold(level: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    without_bp = without(LEVEL_COLUMNS[level], BLOOD_PRESSURE)  # already out of the final lists
    risk = pd.DataFrame(index=train.index, columns=OPTIONS, dtype=float)
    age = risk.copy()
    for fit_rows, validation_rows in folds:
        fit, validation = train.iloc[fit_rows], train.iloc[validation_rows]
        hidden = validation.assign(**{column: np.nan for column in BLOOD_PRESSURE})
        with_bp = build_pipeline("cox", without_bp + BLOOD_PRESSURE).fit(fit, survival_target(fit))
        no_bp = build_pipeline("cox", without_bp).fit(fit, survival_target(fit))
        for option, model, people in [("real", with_bp, validation), ("filled", with_bp, hidden), ("removed", no_bp, validation)]:
            reference = fit_reference(risk_at_horizon(model, fit), fit)
            risk.loc[validation.index, option] = risk_at_horizon(model, people)
            age.loc[validation.index, option] = biological_age(risk.loc[validation.index, option], validation["sex"], reference)
    return risk, age


results = {level: out_of_fold(level) for level in LEVELS}

## 2. ¿Ordena peor a personas de la misma edad?

In [4]:
rows = []
for level, (risk, _) in results.items():
    for option in OPTIONS:
        calibration = calibration_table(risk[option], train)
        rows.append({
            "level": level,
            "option": option,
            "c_index_same_age": same_age_c_index(c_index_by_group(risk[option], train, age_groups(train))),
            "calibration_gap": calibration["gap"].abs().mean(),
            "worst_underestimate": calibration["gap"].max(),
        })
summary = pd.DataFrame(rows).set_index(["level", "option"])
summary["change"] = summary["c_index_same_age"] - summary["c_index_same_age"].groupby("level").transform("first")
summary.round(4)

c_index_same_age  calibration_gap  worst_underestimate  change
level  option                                                                 
extras real               0.7205           0.0050               0.0106  0.0000
       filled             0.7208           0.0053               0.0149  0.0004
       removed            0.7208           0.0052               0.0114  0.0004
full   real               0.7385           0.0055               0.0184  0.0000
       filled             0.7388           0.0058               0.0223  0.0004
       removed            0.7389           0.0057               0.0188  0.0004

## 3. ¿Cuánto cambia la edad biológica?

Diferencia con la edad que daría el modelo conociendo la tensión, solo en quienes la tienen medida. Por debajo de 0, la opción sale **más joven** de la cuenta, que es el error peor.

In [5]:
measured = train[BLOOD_PRESSURE].notna().all(axis=1)
systolic = train.loc[measured, "systolic_bp"]
bands = pd.cut(systolic, [0, 120, 140, 160, 300], right=False, labels=["<120", "120-139", "140-159", "160+"])

tables = {}
for level, (_, age) in results.items():
    for option in ["filled", "removed"]:
        change = (age[option] - age["real"])[measured]
        tables[(level, option)] = pd.concat([
            pd.Series({"all: median of |change|": change.abs().median(), "all: p90 of |change|": change.abs().quantile(0.9)}),
            change.groupby(bands, observed=True).mean().rename(lambda band: f"systolic {band}: mean change"),
        ])
years = pd.DataFrame(tables)
years["share of people"] = [np.nan, np.nan, *bands.value_counts(normalize=True, sort=False)]
years.round(2)

extras           full         share of people
                              filled removed filled removed                
all: median of |change|         0.18    0.19   0.16    0.17             NaN
all: p90 of |change|            0.57    0.61   0.51    0.54             NaN
systolic <120: mean change      0.02    0.04   0.07    0.09            0.49
systolic 120-139: mean change  -0.02   -0.01  -0.05   -0.02            0.35
systolic 140-159: mean change  -0.11   -0.09  -0.21   -0.15            0.12
systolic 160+: mean change     -0.31   -0.22  -0.50   -0.36            0.04

## 4. ¿Representa el HDL al resto del colesterol?

Para decidir si la fila del resultado puede llamarse "Colesterol" saliendo solo del HDL. Correlación de Spearman, y cuánta gente con el HDL bien tiene alto el no-HDL (160 mg/dL o más), que es el que sube el riesgo.

In [6]:
LIPIDS = ["total_cholesterol", "non_hdl", "triglycerides"]
print(train[["hdl", *LIPIDS]].corr(method="spearman")["hdl"].round(2).to_string())

known = train[train[["hdl", "non_hdl"]].notna().all(axis=1)]
good_hdl = known["hdl"] >= known["sex"].map({"male": 40, "female": 50}).astype(float)
high_non_hdl = known["non_hdl"] >= 160
pd.DataFrame({"people": good_hdl.map({True: "good HDL", False: "low HDL"}).value_counts(),
              "high non-HDL": high_non_hdl.groupby(good_hdl.map({True: "good HDL", False: "low HDL"})).mean().round(3)})

hdl                  1.00
total_cholesterol    0.13
non_hdl             -0.23
triglycerides       -0.48


,people,high non-HDL
good HDL,25953,0.257
low HDL,12314,0.388


## 5. Conclusión

**Ni quitar ni rellenar la tensión empeora el modelo.** El C-index misma edad cambia +0,0004 en los dos niveles, lejos del listón de −0,002 (sin bootstrap, así que tampoco es una mejora).

| | `filled` | `removed` |
|---|---|---|
| Cambio típico de la edad (mediana) | 0,16 a 0,18 años | 0,17 a 0,19 años |
| Cambio en 9 de cada 10 personas | hasta 0,57 años | hasta 0,61 años |
| Sistólica de 160 o más (4 % de la gente), de media | de 0,3 a 0,5 años más joven | de 0,2 a 0,4 años más joven |

El error va en la dirección mala, pero es pequeño.

**El HDL no representa al resto del colesterol.** Su correlación es de 0,13 con el total, −0,23 con el no-HDL y −0,48 con los triglicéridos. Con el HDL bien, una de cada cuatro personas (26 %) tiene alto el no-HDL; con el HDL bajo, el 39 %.

**Decisión:** se quita de los dos niveles. La app no la tendría casi nunca, así que se rellenaría siempre.